In [36]:
import numpy as np

import random_forest_oconnell
import pandas as pd
from sklearn.ensemble import  RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [20]:
# CONSTANTS
FEATURE_COLS = [
    "danceability",
    "energy",
    "tempo",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "loudness",
    "key",
    "mode"
]

REQUIRED_COLS = ["artist", "track"] + FEATURE_COLS
# Create df from big dataset and clean it so cols match and it only has one artist
SONGS = pd.read_csv("data/tracks_features.csv")
SONGS = SONGS.rename(columns={"name": "track"})
SONGS["artist"] = SONGS["artists"].apply(random_forest_oconnell.extract_first_artist)
SONGS = SONGS[REQUIRED_COLS]
SONGS["artist"] = SONGS["artist"].astype(str).str.strip()
SONGS["track"] = SONGS["track"].astype(str).str.strip()

In [21]:
# Loading user data from csv
song_df = pd.read_csv("data/tre_lastfm_new.csv", sep=",")
song_df.columns = song_df.columns.str.strip().str.lower()
song_df["count"] = 1
song_df = (
    song_df.groupby(["artist", "track"], as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
if len(song_df) > 200:
    song_df = song_df.iloc[:200]    # Trimming df if it's longer than 200 values
features = random_forest_oconnell.obtain_features(song_df)

Searching Spotify for IDs...
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200
Fetched features for batch 0 (40 songs)
Fetched features for batch 0 (40 songs)
Fetched features for batch 1 (40 songs)
Fetched features for batch 2 (40 songs)
Fetched features for batch 3 (40 songs)


In [22]:
song_df_with_feats = random_forest_oconnell.create_feature_df(song_df, features)

In [23]:
# Saving it in case I need to use it in the future
song_df_with_feats.to_csv("data/tre_lastfm_new_with_features.csv", index=False)

In [24]:
#Dropping rows with NA values
clean_df = song_df_with_feats.dropna(subset=FEATURE_COLS).copy()

In [25]:
# Creating user profile so we can see what they generally like
user_profile = (
    clean_df[FEATURE_COLS]
    .multiply(clean_df["count"], axis=0)
    .sum() / clean_df["count"].sum()
)

print(user_profile)

danceability          0.607069
energy                0.620782
tempo               124.461759
valence               0.419092
acousticness          0.192614
instrumentalness      0.076739
liveness              0.215944
speechiness           0.091657
loudness             -6.541287
key                   4.891089
mode                  0.597360
dtype: float64


In [26]:
# Now I'm going to remove songs that the user has listened to, so we don't recommend something they've heard already.
heard_songs = clean_df[["artist", "track"]].drop_duplicates().copy()

candidate_df = SONGS.merge(
    heard_songs,
    on=["artist", "track"],
    how="left",
    indicator=True
)
candidate_df = candidate_df[candidate_df["_merge"] == "left_only"].drop(columns=["_merge"])

In [27]:
# Inserting some unheard songs, so the model can better predict.
positive_df = clean_df.copy()
negative_df = candidate_df.sample(n=int(len(positive_df))).copy()
negative_df["count"] = 0

train_df = pd.concat([positive_df, negative_df], ignore_index=True)

X = train_df[FEATURE_COLS]
y = train_df["count"]

In [28]:
# Actually training the model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Model evaluation:")
mse = mean_squared_error(y_test, pred)
print("MSE:", mse)

Model evaluation:
MSE: 4.092575


In [29]:
# Looping to get Average MSE over X iterations:
mse_sum = 0
for i in range(50):
    positive_df = clean_df.copy()
    negative_df = candidate_df.sample(n=int(len(positive_df))).copy()
    negative_df["count"] = 0

    train_df = pd.concat([positive_df, negative_df], ignore_index=True)

    X = train_df[FEATURE_COLS]
    y = train_df["count"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2
    )

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mse_sum += mean_squared_error(y_test, pred)

mse_avg = mse_sum / 50
print("MSE:", mse_avg)

MSE: 4.20841229245283


In [37]:
print("RMSE:", np.sqrt(mse_avg))

RMSE: 2.051441515728106


In [30]:
# For reference, here's the model's performance without adding false/noisy values
X_no_noise = clean_df[FEATURE_COLS]
y_no_noise = clean_df["count"]
# Actually training the model
X_train_no_noise, X_test_no_noise, y_train_no_noise, y_test_no_noise = train_test_split(
    X_no_noise, y_no_noise, test_size=0.2
)

model_with_no_noise = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model_with_no_noise.fit(X_train_no_noise, y_train_no_noise)

pred_no_noise = model_with_no_noise.predict(X_test_no_noise)

print("Model evaluation:")
mse = mean_squared_error(y_test_no_noise, pred_no_noise)
print("MSE:", mse)

Model evaluation:
MSE: 16.103835185185183


In [31]:
# Getting recommendations
candidate_X = candidate_df[FEATURE_COLS]
candidate_df["predicted_count"] = model.predict(candidate_X)
recommendations = candidate_df.sort_values(
    "predicted_count",
    ascending=False
)

In [32]:
recommendations.head(20)

,artist,track,danceability,energy,tempo,valence,acousticness,instrumentalness,liveness,speechiness,loudness,key,mode,predicted_count
322777,Wiwek,Run,0.687,0.931,125.978,0.0444,0.003160,0.962000,0.633,0.0688,-5.006,11,0,9.130
258525,Kelly Clarkson,Ready,0.673,0.868,122.085,0.4440,0.014500,0.000000,0.612,0.0728,-4.147,11,1,8.900
105150,Ria Mae,Clothes Off - Glenn Morrison Commercial Remix,0.692,0.725,120.052,0.6260,0.060000,0.000048,0.605,0.0505,-2.884,11,0,8.640
1116861,Ed:it,Replicants - Original Mix,0.688,0.972,172.021,0.4170,0.000577,0.859000,0.614,0.1570,-3.176,11,1,8.605
1069603,King Nun,Low Flying Dandelion,0.691,0.820,115.008,0.7600,0.000608,0.000228,0.601,0.0766,-4.190,8,0,8.560
813813,K-Syran,Shake That Booty (Electrik Disco Remix),0.689,0.874,127.987,0.7400,0.000106,0.094700,0.629,0.0466,-4.720,9,0,8.510
944449,Julian Calor,Dream Odyssey,0.687,0.851,134.006,0.0614,0.037200,0.891000,0.608,0.0558,-7.083,8,1,8.450
984999,Keith Kenny,Constellation Conversation,0.690,0.841,120.146,0.5230,0.042800,0.000939,0.608,0.0472,-5.211,10,0,8.405
565427,Atlanta Rhythm Section,Who You Gonna Run To,0.691,0.724,109.959,0.6620,0.011100,0.003170,0.624,0.0261,-4.922,9,1,8.400
1095852,Roma Pafos,One Shot - Thomas Penton Remix,0.675,0.963,127.976,0.4040,0.018600,0.003920,0.621,0.0485,-4.166,0,1,8.385


In [33]:
# Demonstrating why I can't just run this on one song :(
single_song_df = SONGS.sample(n=1, random_state=42).copy()
single_song_df["count"] = 1
single_song_df

,artist,track,danceability,energy,tempo,valence,acousticness,instrumentalness,liveness,speechiness,loudness,key,mode,count
54449,Unk,Smokin' Sticky Sticky,0.623,0.736,87.988,0.422,0.0021,0.0,0.0691,0.402,-3.657,11,0,1


In [34]:
X_single_song = single_song_df[FEATURE_COLS]
y_single_song = single_song_df["count"]

single_song_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

single_song_model.fit(X_single_song, y_single_song)

candidate_df_single_song = SONGS.copy()
candidate_X_single_song = candidate_df_single_song[FEATURE_COLS]

candidate_df_single_song["predicted_count"] = single_song_model.predict(candidate_X_single_song)

recommendations = candidate_df_single_song.sort_values(
    "predicted_count",
    ascending=False
)

recommendations.head(20)

,artist,track,danceability,energy,tempo,valence,acousticness,instrumentalness,liveness,speechiness,loudness,key,mode,predicted_count
0,Rage Against The Machine,Testify,0.470,0.9780,117.906,0.5030,0.026100,0.000011,0.3560,0.0727,-5.399,7,1,1.0
802672,Tragederia,The Architect (Single Version),0.541,0.9690,104.016,0.0838,0.000971,0.703000,0.0908,0.0587,-5.297,2,1,1.0
802688,Martin O'Donnell,We're Not Going Anywhere,0.199,0.1380,66.964,0.0391,0.280000,0.877000,0.0927,0.0403,-21.405,3,1,1.0
802687,Martin O'Donnell,Fortress,0.770,0.7700,107.041,0.3510,0.088200,0.954000,0.2100,0.0663,-16.270,1,1,1.0
802686,Martin O'Donnell,Ashes,0.241,0.0850,82.756,0.0433,0.984000,0.968000,0.0986,0.0359,-19.295,5,0,1.0
802685,Martin O'Donnell,From the Vault,0.729,0.3930,102.968,0.4110,0.671000,0.905000,0.0961,0.0290,-19.096,6,1,1.0
802684,Martin O'Donnell,Epilogue,0.104,0.0767,67.554,0.0354,0.966000,0.958000,0.0930,0.0398,-18.684,8,1,1.0
802683,Martin O'Donnell,The Pillar of Autumn,0.322,0.4340,103.054,0.1280,0.288000,0.849000,0.1010,0.0717,-17.167,5,0,1.0
802682,Martin O'Donnell,The Package,0.276,0.2990,63.813,0.0352,0.663000,0.871000,0.0857,0.0497,-16.743,5,1,1.0
802681,Martin O'Donnell,New Alexandria,0.183,0.2650,80.417,0.0399,0.838000,0.864000,0.1100,0.0423,-19.500,1,1,1.0
